In [1]:
import torch
import torchgeo
from torch.utils.data import DataLoader
import torch.nn as nn
from torch import Tensor
from torch.nn.modules import Module
import torch.optim as optim

import torchgeo
from torchgeo.datasets import RasterDataset, Sentinel2, stack_samples
import torchgeo.models
from torchgeo.samplers import RandomGeoSampler, RandomBatchGeoSampler,  GridGeoSampler
from torchgeo.samplers.constants import Units
from torchgeo.datamodules import GeoDataModule

import utils
from importlib import reload

In [2]:
# todo: wrap train as pytorch lightning tasks
nir_band = 3
label_band = 7
vis_band_start = 4
vis_band_end = 7

def my_transforms(sample):
    
    sample['image'][0] = (sample['image'][0] - 1400.) / 60.
    sample['image'][1] = (sample['image'][1] - 1600.) / 60.
    sample['image'][2] = (sample['image'][2] - 1600.) / 120.
    sample['image'][nir_band] = (sample['image'][nir_band] - 3000.) / 300.
    
    sample['vis'] = sample['image'][vis_band_start:vis_band_end]#.detach().clone()
    
    # package for dataloader
    
    # last band for label
    label_band = len(sample['image']) - 1 
    sample['mask'] = torch.Tensor(sample['image'][label_band]) #.detach().clone()
    
    # 4 channel image
    sample['image'] = sample['image'][:4]#.detach().clone()
    
    return sample



In [3]:
def get_chm_sites(sites=[6,13,14],
                  layers=['r','g','b','nir','vis','chm']):
    
    assert len(sites) > 0
    
    # first site
    chms = utils.make_site_dataset(sites[0],
                                   transforms=my_transforms,
                                               layers=layers, 
                                               data_dir='data')
        
    if len(sites) > 1:
        for site in sites[1:]:
            chm_this = utils.make_site_dataset(site,
                                              transforms=my_transforms,
                                               layers=layers, 
                                               data_dir='data',
                                              )
            
            chms = UnionDataset(chms, chm_this)
            
    return chms

In [4]:
from typing import Any, Optional, Dict
import matplotlib.pyplot as plt

class ChmDataModule(GeoDataModule):
    def __init__(
        self,
        train_sites: list[str],
        val_sites: list[str],
        test_sites: list[str],
        layers: list[str] = ['r','g','b','nir','vis'],
        batch_size: int = 8,
        patch_size: int = 64,
        length: int = 1000,
        num_workers: int = 0,
        **kwargs: Any,
    ) -> None:
        
        super().__init__(
            get_chm_sites, batch_size, patch_size, length, num_workers, **kwargs
        )

        self.train_sites = train_sites
        self.val_sites = val_sites
        self.test_sites = test_sites
        self.layers = layers
        self.original_patch_size = self.patch_size
        
        self.plt_vmax = 15

    def setup(self, stage: str) -> None:
        if stage in ["fit"]:
            self.train_dataset = get_chm_sites(sites=self.train_sites, layers=self.layers, **self.kwargs)
            self.train_batch_sampler = RandomBatchGeoSampler(
                self.train_dataset,
                self.original_patch_size,
                self.batch_size,
                self.length,
            )
        if stage in ["fit", "validate"]:
            self.val_dataset = get_chm_sites(
                sites=self.val_sites, layers=self.layers, **self.kwargs
            )
            self.val_sampler = GridGeoSampler(
                self.val_dataset, self.original_patch_size, self.original_patch_size
            )
        if stage in ["test"]:
            self.test_dataset = get_chm_sites(
                sites=self.test_sites, layers=self.layers, **self.kwargs
            )
            self.test_sampler = GridGeoSampler(
                self.test_dataset, self.original_patch_size, self.original_patch_size
            )
            
    def plot(
        self,
        sample: Dict[str, Any],
        show_titles: bool = True,
        suptitle: Optional[str] = None,
    ) -> plt.Figure:
        """Plot a sample from the dataset.

        Args:
            sample: a sample returned by :meth:`RasterDataset.__getitem__`
            show_titles: flag indicating whether to show titles above each panel
            suptitle: optional suptitle to use for figure

        Returns:
            a matplotlib Figure with the rendered sample

        .. versionchanged:: 0.3
           Method now takes a sample dict, not a Tensor. Additionally, possible to
           show subplot titles and/or use a custom suptitle.
        """
        mask = sample["mask"].squeeze(0).cpu().numpy()
        nan_mask = mask < 0
        ncols = 2
        
        nan_plot_val = 0
        mask[nan_mask] = nan_plot_val
        
        vis = sample["vis"].cpu().numpy() / 255.
        
        for c in range(vis.shape[0]):
            vis[c][nan_mask] = nan_plot_val
            
        showing_predictions = "prediction" in sample
        if showing_predictions:
            pred = sample["prediction"].squeeze(0).cpu().numpy()
            ncols = 3

            pred[nan_mask] = nan_plot_val
            
        fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(4 * ncols, 4))

        if showing_predictions:
            axs[0].imshow(
                vis.transpose(1,2,0),
                interpolation="none",
            )
            axs[1].axis("off")
            axs[1].imshow(
                mask,
                vmin=0,
                vmax=self.plt_vmax,
                cmap='Greens',
                interpolation="none",
            )
            axs[1].axis("off")
            axs[2].imshow(
                pred,
                vmin=0,
                vmax=self.plt_vmax,
                cmap='Greens',
                interpolation="none",
            )
            axs[2].axis("off")
            if show_titles:
                axs[0].set_title("Img (3 channel)")
                axs[1].set_title("Mask")
                axs[2].set_title("Prediction")

        else:
            axs[0].imshow(
                vis.transpose(1,2,0),
                interpolation="none",
            )
            axs[1].imshow(
                mask,
                vmin=0,
                vmax=self.plt_vmax,
                cmap='Greens',
                interpolation="none",
            )
            axs[1].axis("off")
            if show_titles:
                axs[0].set_title("Img (3 channel)")
                axs[1].set_title("Mask")

        if suptitle is not None:
            plt.suptitle(suptitle)

        return fig
            


In [5]:
chm = ChmDataModule(train_sites=[6], val_sites=[13],test_sites=[14])

In [6]:
import pixelwise_regression_task_with_mask
reload(pixelwise_regression_task_with_mask)
#from  pixelwise_regression_task import PixelwiseRegressionTask

<module 'pixelwise_regression_task_with_mask' from '/n/home10/erolf/tree_mapping/pixelwise_regression_task_with_mask.py'>

In [17]:
task = pixelwise_regression_task_with_mask.PixelwiseRegressionTask(model='fcn', 
                               loss='mse',
                               learning_rate=0.001,
                               learning_rate_schedule_patience=10,
                               weights=None,
                                in_channels=4,
                                num_classes=1, 
                                num_filters=128,
                        )

In [18]:
import os
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
default_root_dir = os.path.join("exp1")
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss", dirpath=default_root_dir, save_top_k=1, save_last=True
)


In [19]:
early_stopping_callback = EarlyStopping(monitor="val_loss", min_delta=0.00, patience=20)
logger = TensorBoardLogger(save_dir=default_root_dir, name="logs")

batch_size = 128
num_workers = 10
max_epochs = 50
fast_dev_run = False

In [20]:
trainer = Trainer(
    accelerator=accelerator,
    callbacks=[checkpoint_callback, 
               #early_stopping_callback
              ],
    fast_dev_run=fast_dev_run,
    log_every_n_steps=1,
    logger=logger,
    min_epochs=1,
    max_epochs=max_epochs,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [21]:
trainer.fit(model=task, datamodule=chm)


6
Converting RasterDataset resolution from 9.995824634655532 to 10


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params
---------------------------------------------------
0 | model         | FCN              | 595 K 
1 | loss          | MSELoss          | 0     
2 | train_metrics | MetricCollection | 0     
3 | val_metrics   | MetricCollection | 0     
4 | test_metrics  | MetricCollection | 0     
---------------------------------------------------
595 K     Trainable params
0         Non-trainable params
595 K     Total params
2.381     Total estimated model params size (MB)


13
Converting RasterDataset resolution from 10.004158004158004 to 10


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=50` reached.
